# Conditional Medical Image Synthesis: GAN vs DDPM

**Course**: CSYE 7374 — Deep Learning and Generative AI in Healthcare

This notebook implements and compares two paradigms for **conditional** medical image generation:

| Model | Mechanism | Strength |
|---|---|---|
| **Conditional GAN** | Adversarial training | Fast inference, sharp textures |
| **Conditional DDPM** | Iterative denoising | Stable training, high diversity |

**Dataset**: OrganAMNIST — 28 × 28 grayscale abdominal CT slices, 11 organ classes
**Application**: Synthetic organ CT generation for rare-class data augmentation

In [ ]:
!pip install -q medmnist

import math, warnings
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision.transforms as transforms
from tqdm import tqdm
import medmnist
from medmnist import INFO

warnings.filterwarnings('ignore')
plt.rcParams['figure.dpi'] = 100
print(f'PyTorch {torch.__version__}')

In [ ]:
class Config:
    DATA_FLAG  = 'organamnist'
    IMG_SIZE   = 28
    N_CH       = 1
    N_CLASSES  = 11
    BATCH_SIZE = 256
    SEED       = 42
    # cGAN
    LATENT_DIM = 64
    GAN_LR     = 2e-4
    GAN_EPOCHS = 60
    # DDPM
    T          = 400
    BETA_START = 1e-4
    BETA_END   = 0.02
    DDPM_LR    = 5e-4
    DDPM_EPOCHS= 60
    TIME_DIM   = 128

device = (torch.device('cuda')  if torch.cuda.is_available() else
          torch.device('mps')   if torch.backends.mps.is_available() else
          torch.device('cpu'))
torch.manual_seed(Config.SEED); np.random.seed(Config.SEED)
print(f'Device: {device}')

CLASS_NAMES = list(INFO[Config.DATA_FLAG]['label'].values())
print(f'Classes: {CLASS_NAMES}')

## 1. Data Loading — OrganAMNIST

In [ ]:
DataClass = getattr(medmnist, INFO[Config.DATA_FLAG]['python_class'])
tfm = transforms.Compose([transforms.ToTensor(), transforms.Normalize([0.5], [0.5])])

train_ds = DataClass(split='train', transform=tfm, download=True)
val_ds   = DataClass(split='val',   transform=tfm, download=True)

train_loader = DataLoader(train_ds, batch_size=Config.BATCH_SIZE,
                          shuffle=True,  num_workers=0, drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=Config.BATCH_SIZE,
                          shuffle=False, num_workers=0)
print(f'Train: {len(train_ds)} | Val: {len(val_ds)}')

In [ ]:
fig, axes = plt.subplots(2, Config.N_CLASSES, figsize=(16, 4))
for cls in range(Config.N_CLASSES):
    idxs = [i for i, (_, l) in enumerate(train_ds) if int(l) == cls][:2]
    for row in range(2):
        img, _ = train_ds[idxs[row]]
        axes[row, cls].imshow(img.squeeze(), cmap='gray', vmin=-1, vmax=1)
        axes[row, cls].axis('off')
        if row == 0:
            axes[row, cls].set_title(CLASS_NAMES[cls], fontsize=7)
plt.suptitle('OrganAMNIST — Real Samples', fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()

## 2. Part 1 — Conditional GAN (cGAN)

**Generator** $G(z, c)$: noise $z \in \mathbb{R}^{64}$ + class embedding $c$ → 28×28 CT slice
**Discriminator** $D(x, c)$: (image, class) pair → real / fake logit

Loss:
$$\mathcal{L}_D = -\mathbb{E}[\log D(x,c)] - \mathbb{E}[\log(1-D(G(z,c),c))]$$
$$\mathcal{L}_G = -\mathbb{E}[\log D(G(z,c),c)]$$

In [ ]:
class Generator(nn.Module):
    def __init__(self):
        super().__init__()
        self.class_emb = nn.Embedding(Config.N_CLASSES, 16)
        self.fc = nn.Sequential(
            nn.Linear(Config.LATENT_DIM + 16, 256 * 7 * 7), nn.ReLU(True))
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(256, 128, 4, 2, 1), nn.BatchNorm2d(128), nn.ReLU(True),
            nn.ConvTranspose2d(128, 64,  4, 2, 1), nn.BatchNorm2d(64),  nn.ReLU(True),
            nn.Conv2d(64, Config.N_CH, 3, 1, 1), nn.Tanh())

    def forward(self, z, c):
        x = self.fc(torch.cat([z, self.class_emb(c)], 1)).view(-1, 256, 7, 7)
        return self.decoder(x)


class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.class_emb = nn.Embedding(Config.N_CLASSES, Config.IMG_SIZE ** 2)
        self.net = nn.Sequential(
            nn.Conv2d(Config.N_CH + 1, 64,  4, 2, 1), nn.LeakyReLU(0.2, True),
            nn.Conv2d(64, 128, 4, 2, 1), nn.BatchNorm2d(128), nn.LeakyReLU(0.2, True),
            nn.Flatten(), nn.Linear(128 * 7 * 7, 1))

    def forward(self, img, c):
        c_map = self.class_emb(c).view(-1, 1, Config.IMG_SIZE, Config.IMG_SIZE)
        return self.net(torch.cat([img, c_map], 1))


G = Generator().to(device)
D = Discriminator().to(device)
print(f'Generator  params: {sum(p.numel() for p in G.parameters()):,}')
print(f'Discriminator params: {sum(p.numel() for p in D.parameters()):,}')

In [ ]:
opt_G = optim.Adam(G.parameters(), lr=Config.GAN_LR, betas=(0.5, 0.999))
opt_D = optim.Adam(D.parameters(), lr=Config.GAN_LR, betas=(0.5, 0.999))
bce   = nn.BCEWithLogitsLoss()
gan_history = {'d': [], 'g': []}

for epoch in range(1, Config.GAN_EPOCHS + 1):
    G.train(); D.train()
    d_sum = g_sum = 0.0
    for imgs, labels in train_loader:
        imgs   = imgs.to(device)
        labels = labels.squeeze().long().to(device)
        B = imgs.size(0)
        real = torch.ones(B, 1, device=device)
        fake = torch.zeros(B, 1, device=device)

        z     = torch.randn(B, Config.LATENT_DIM, device=device)
        fakes = G(z, labels).detach()
        opt_D.zero_grad()
        loss_D = 0.5 * (bce(D(imgs, labels), real) + bce(D(fakes, labels), fake))
        loss_D.backward(); opt_D.step()

        z     = torch.randn(B, Config.LATENT_DIM, device=device)
        fakes = G(z, labels)
        opt_G.zero_grad()
        loss_G = bce(D(fakes, labels), real)
        loss_G.backward(); opt_G.step()

        d_sum += loss_D.item(); g_sum += loss_G.item()

    gan_history['d'].append(d_sum / len(train_loader))
    gan_history['g'].append(g_sum / len(train_loader))
    if epoch % 10 == 0:
        print(f'Epoch {epoch:3d} | D: {gan_history["d"][-1]:.4f}  G: {gan_history["g"][-1]:.4f}')

In [ ]:
G.eval()
fig, axes = plt.subplots(2, Config.N_CLASSES, figsize=(16, 4))
with torch.no_grad():
    for cls in range(Config.N_CLASSES):
        z   = torch.randn(2, Config.LATENT_DIM, device=device)
        lbl = torch.full((2,), cls, dtype=torch.long, device=device)
        out = G(z, lbl).cpu()
        for row in range(2):
            axes[row, cls].imshow(out[row].squeeze(), cmap='gray', vmin=-1, vmax=1)
            axes[row, cls].axis('off')
            if row == 0:
                axes[row, cls].set_title(CLASS_NAMES[cls], fontsize=7)
plt.suptitle('cGAN — Generated Samples (2 per class)', fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()

## 3. Part 2 — Denoising Diffusion Probabilistic Model (DDPM)

**Forward process** (data → noise):
$$q(x_t | x_0) = \mathcal{N}\!\left(\sqrt{\bar\alpha_t}\, x_0,\; (1-\bar\alpha_t)\mathbf{I}\right)$$

**Training** — predict added noise $\epsilon$ given noisy image $x_t$, timestep $t$, class $c$:
$$\mathcal{L} = \mathbb{E}_{x_0,\epsilon,t}\left[\|\epsilon - \epsilon_\theta(x_t, t, c)\|^2\right]$$

**Reverse sampling** (noise → image, $T$ steps):
$$x_{t-1} = \frac{1}{\sqrt{\alpha_t}}\!\left(x_t - \frac{1-\alpha_t}{\sqrt{1-\bar\alpha_t}}\,\epsilon_\theta(x_t,t,c)\right) + \sqrt{\beta_t}\,z$$

In [ ]:
betas     = torch.linspace(Config.BETA_START, Config.BETA_END, Config.T, device=device)
alphas    = 1.0 - betas
alpha_bar = torch.cumprod(alphas, dim=0)
sqrt_ab   = alpha_bar.sqrt()
sqrt_1mab = (1.0 - alpha_bar).sqrt()

def extract(a, t, shape):
    v = a.gather(0, t)
    return v.reshape(t.shape[0], *((1,) * (len(shape) - 1)))

def q_sample(x0, t, noise):
    return extract(sqrt_ab, t, x0.shape) * x0 + extract(sqrt_1mab, t, x0.shape) * noise

In [ ]:
class SinEmb(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim
    def forward(self, t):
        half = self.dim // 2
        freq = torch.exp(-math.log(10000) * torch.arange(half, device=t.device) / (half - 1))
        emb  = t[:, None].float() * freq[None, :]
        return torch.cat([emb.sin(), emb.cos()], dim=-1)


class DBlock(nn.Module):
    def __init__(self, in_ch, out_ch, tdim):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch,  out_ch, 3, 1, 1)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, 1, 1)
        self.gn1   = nn.GroupNorm(8, out_ch)
        self.gn2   = nn.GroupNorm(8, out_ch)
        self.temb  = nn.Linear(tdim, out_ch)
        self.cemb  = nn.Linear(tdim, out_ch)
    def forward(self, x, te, ce):
        h = F.silu(self.gn1(self.conv1(x)))
        h = h + self.temb(te)[:, :, None, None] + self.cemb(ce)[:, :, None, None]
        return F.silu(self.gn2(self.conv2(h)))


class UNetDDPM(nn.Module):
    def __init__(self):
        super().__init__()
        td = Config.TIME_DIM
        self.time_mlp  = nn.Sequential(SinEmb(td), nn.Linear(td, td), nn.SiLU())
        self.class_emb = nn.Embedding(Config.N_CLASSES, td)
        self.e1  = DBlock(Config.N_CH, 64,  td)
        self.e2  = DBlock(64,  128, td)
        self.bn  = DBlock(128, 256, td)
        self.d1  = DBlock(256 + 128, 128, td)
        self.d2  = DBlock(128 + 64,  64,  td)
        self.out = nn.Conv2d(64, Config.N_CH, 1)
        self.pool = nn.MaxPool2d(2)
        self.up1  = nn.ConvTranspose2d(256, 256, 2, 2)
        self.up2  = nn.ConvTranspose2d(128, 128, 2, 2)

    def forward(self, x, t, c):
        te = self.time_mlp(t)
        ce = self.class_emb(c)
        e1 = self.e1(x,           te, ce)
        e2 = self.e2(self.pool(e1), te, ce)
        b  = self.bn(self.pool(e2), te, ce)
        d1 = self.d1(torch.cat([self.up1(b),  e2], 1), te, ce)
        d2 = self.d2(torch.cat([self.up2(d1), e1], 1), te, ce)
        return self.out(d2)


unet = UNetDDPM().to(device)
print(f'UNet params: {sum(p.numel() for p in unet.parameters()):,}')

In [ ]:
opt_ddpm     = optim.Adam(unet.parameters(), lr=Config.DDPM_LR)
ddpm_history = []

for epoch in range(1, Config.DDPM_EPOCHS + 1):
    unet.train()
    ep_loss = 0.0
    for imgs, labels in train_loader:
        imgs   = imgs.to(device)
        labels = labels.squeeze().long().to(device)
        noise  = torch.randn_like(imgs)
        t = torch.randint(0, Config.T, (imgs.size(0),), device=device, dtype=torch.long)
        xt   = q_sample(imgs, t, noise)
        pred = unet(xt, t, labels)
        loss = F.mse_loss(pred, noise)
        opt_ddpm.zero_grad(); loss.backward(); opt_ddpm.step()
        ep_loss += loss.item()
    ddpm_history.append(ep_loss / len(train_loader))
    if epoch % 10 == 0:
        print(f'Epoch {epoch:3d} | Loss: {ddpm_history[-1]:.5f}')

In [ ]:
@torch.no_grad()
def ddpm_sample(class_list):
    unet.eval()
    n = len(class_list)
    x = torch.randn(n, Config.N_CH, Config.IMG_SIZE, Config.IMG_SIZE, device=device)
    c = torch.tensor(class_list, dtype=torch.long, device=device)
    for i in tqdm(reversed(range(Config.T)), total=Config.T, desc='Sampling', leave=False):
        t   = torch.full((n,), i, device=device, dtype=torch.long)
        eps = unet(x, t, c)
        b_t = betas[i]; a_t = alphas[i]; ab_t = alpha_bar[i]
        mean = (1.0 / a_t.sqrt()) * (x - b_t / (1.0 - ab_t).sqrt() * eps)
        x = mean if i == 0 else mean + b_t.sqrt() * torch.randn_like(x)
    return x.clamp(-1, 1)


samples = ddpm_sample(list(range(Config.N_CLASSES)) * 2)
fig, axes = plt.subplots(2, Config.N_CLASSES, figsize=(16, 4))
for cls in range(Config.N_CLASSES):
    for row in range(2):
        axes[row, cls].imshow(samples[cls + row * Config.N_CLASSES].cpu().squeeze(),
                              cmap='gray', vmin=-1, vmax=1)
        axes[row, cls].axis('off')
        if row == 0:
            axes[row, cls].set_title(CLASS_NAMES[cls], fontsize=7)
plt.suptitle('DDPM — Generated Samples (2 per class)', fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()

## 4. Comparison

In [ ]:
# --- Training curves ---
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(gan_history['d'], label='Discriminator')
axes[0].plot(gan_history['g'], label='Generator')
axes[0].set_title('cGAN Training Loss'); axes[0].set_xlabel('Epoch')
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(ddpm_history, color='steelblue', label='Noise MSE')
axes[1].set_title('DDPM Training Loss'); axes[1].set_xlabel('Epoch')
axes[1].legend(); axes[1].grid(alpha=0.3)
plt.suptitle('Training Dynamics', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# --- Side-by-side: Real | GAN x2 | DDPM x2 for 4 representative organs ---
show_cls = [3, 4, 6, 9]   # heart, kidney-l, liver, spleen
N = len(show_cls)

G.eval()
with torch.no_grad():
    lbl_t     = torch.tensor([c for c in show_cls for _ in range(2)],
                              dtype=torch.long, device=device)
    z         = torch.randn(N * 2, Config.LATENT_DIM, device=device)
    gan_out   = G(z, lbl_t).cpu()
ddpm_out = ddpm_sample([c for c in show_cls for _ in range(2)]).cpu()

fig, axes = plt.subplots(N, 5, figsize=(10, N * 2.2))
for col, title in enumerate(['Real', 'GAN (1)', 'GAN (2)', 'DDPM (1)', 'DDPM (2)']):
    axes[0, col].set_title(title, fontsize=10, fontweight='bold')

for row, cls in enumerate(show_cls):
    real_idx = next(i for i, (_, l) in enumerate(train_ds) if int(l) == cls)
    real, _  = train_ds[real_idx]
    srcs = [real, gan_out[row * 2], gan_out[row * 2 + 1],
                   ddpm_out[row * 2], ddpm_out[row * 2 + 1]]
    axes[row, 0].set_ylabel(CLASS_NAMES[cls], fontsize=9)
    for col, img in enumerate(srcs):
        axes[row, col].imshow(img.squeeze(), cmap='gray', vmin=-1, vmax=1)
        axes[row, col].axis('off')

plt.suptitle('Real vs cGAN vs DDPM', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

## 5. Discussion

| | cGAN | DDPM |
|---|---|---|
| **Training** | Adversarial — can be unstable (mode collapse) | Stable MSE objective |
| **Inference** | Single forward pass — milliseconds | $T=400$ denoising steps — slow |
| **Sample diversity** | May undercover rare modes | Better coverage of data distribution |
| **Conditioning** | Class embedding injected into $G$ and $D$ | Class embedding in UNet at every level |
| **Clinical use-case** | Fast augmentation pipelines | High-fidelity synthesis when compute allows |

Both models demonstrate **class-conditional organ CT synthesis**, a direct clinical need: training data for rare-organ segmentation models often requires expensive annotation. Generative augmentation with either model can double effective dataset size without additional scans.

**Extensions**: Replace class conditioning with a real paired image (Pix2Pix / DDIB) to perform modality translation (e.g., MRI T1 → T2, CT → MRI), where the conditioning signal is the source-domain image rather than a discrete label.